# 📈 Notebook 5 — Model Evaluation & Economic Impact

## Coal Demand Forecasting — CRISP-ML(Q) Framework

**Objective:** Compare all 4 models, identify the best performer, and quantify economic savings in INR.

### What this notebook covers:
1. **Metrics computation** — MAPE, RMSE, MAE, R² for each model
2. **Baseline Comparison Table** — Side-by-side performance (professor feedback)
3. **Target Achievement Check** — MAPE < 8%, RMSE < 50 tonnes
4. **Best Model Forecast Plot** — Actual vs Predicted visualization
5. **Prediction Intervals** — 95% confidence bands
6. **Economic Impact Analysis** — Cost savings in INR (professor feedback)

---

## 5.1 Setup — Train All Models

In [1]:
import os, sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

from src.config import (
    PROCESSED_TRAIN_FILE, PROCESSED_VAL_FILE, PROCESSED_TEST_FILE,
    TARGET_MAPE, TARGET_RMSE,
    HOLDING_COST_PER_TONNE_PER_DAY, SHORTAGE_COST_PER_MWH,
    AVG_MWH_LOST_PER_SHORTAGE, INVENTORY_REDUCTION_PERCENT, DAYS_PER_YEAR,
    REPORTS_DIR
)

train_df = pd.read_csv(PROCESSED_TRAIN_FILE)
val_df = pd.read_csv(PROCESSED_VAL_FILE)
test_df = pd.read_csv(PROCESSED_TEST_FILE)

print("Loading preprocessed data and training all 4 models...")
print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

Loading preprocessed data and training all 4 models...
Train: (746, 18), Val: (160, 18), Test: (160, 18)


In [2]:
# Train all 4 models
from src.models.arima_model import train_arima
from src.models.prophet_model import train_prophet
from src.models.lstm_model import train_lstm
from src.models.xgboost_model import train_xgboost

print("[1/4] Training ARIMA...")
arima_results = train_arima(train_df, val_df, test_df)
print(f"       ✓ Done in {arima_results['training_time']:.2f}s\n")

print("[2/4] Training Prophet...")
prophet_results = train_prophet(train_df, val_df, test_df)
print(f"       ✓ Done in {prophet_results['training_time']:.2f}s\n")

print("[3/4] Training LSTM...")
lstm_results = train_lstm(train_df, val_df, test_df)
print(f"       ✓ Done in {lstm_results['training_time']:.2f}s\n")

print("[4/4] Training XGBoost...")
xgb_results = train_xgboost(train_df, val_df, test_df)
print(f"       ✓ Done in {xgb_results['training_time']:.2f}s\n")

all_results = [arima_results, prophet_results, lstm_results, xgb_results]
print("✅ All 4 models trained successfully.")

[1/4] Training ARIMA...
[2026-03-02 19:22:51] [INFO] [training] ==================================================


[2026-03-02 19:22:51] [INFO] [training] Training ARIMA model...


[2026-03-02 19:22:51] [INFO] [training] ==================================================


[2026-03-02 19:22:51] [INFO] [training] Training samples: 906, Test samples: 160


[2026-03-02 19:22:51] [INFO] [training] Running auto_arima (this may take a few minutes)...


...

[2026-03-02 19:23:17] [INFO] [training] Feature importance plot saved to /Users/palasudeepkumar/Desktop/Foundation_Project/coal-demand-forecasting/reports/xgb_feature_importance.png


[2026-03-02 19:23:17] [INFO] [training] XGBoost model saved to /Users/palasudeepkumar/Desktop/Foundation_Project/coal-demand-forecasting/models/xgboost_model.pkl


[2026-03-02 19:23:17] [INFO] [training] XGBoost training completed in 4.23s


[2026-03-02 19:23:17] [INFO] [training] XGBoost inference time: 0.24ms for 160 samples


       ✓ Done in 4.23s

✅ All 4 models trained successfully.


## 5.2 Metrics Computation

We evaluate each model using 4 key metrics:

| Metric | Formula | Target |
|--------|---------|--------|
| **MAPE** | Mean Absolute Percentage Error | < 8% |
| **RMSE** | Root Mean Squared Error | < 50 tonnes |
| **MAE** | Mean Absolute Error | Lower is better |
| **R²** | Coefficient of Determination | Closer to 1 is better |

In [3]:
def compute_mape(actuals, predictions):
    mask = actuals != 0
    return float(np.mean(np.abs((actuals[mask] - predictions[mask]) / actuals[mask])) * 100)

def compute_metrics(actuals, predictions):
    mape = compute_mape(actuals, predictions)
    rmse = float(np.sqrt(mean_squared_error(actuals, predictions)))
    mae = float(mean_absolute_error(actuals, predictions))
    r2 = float(r2_score(actuals, predictions))
    return {'MAPE': mape, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

# Compute metrics for all models
rows = []
for result in all_results:
    actuals = result['actuals']
    preds = result['predictions']
    min_len = min(len(actuals), len(preds))
    metrics = compute_metrics(actuals[:min_len], preds[:min_len])
    rows.append({
        'Model': result['model_name'],
        'MAPE %': round(metrics['MAPE'], 2),
        'RMSE': round(metrics['RMSE'], 2),
        'MAE': round(metrics['MAE'], 2),
        'R²': round(metrics['R2'], 4),
        'Train (s)': round(result.get('training_time', 0), 2),
        'Infer (ms)': round(result.get('inference_time_ms', 0), 2),
    })

comparison_df = pd.DataFrame(rows)
print("Metrics computed for all models.")

Metrics computed for all models.


## 5.3 Model Comparison Table (Baseline Comparison)

This is the **baseline comparison table** across all 4 models, addressing professor feedback to include explicit side-by-side comparison.

In [4]:
# Formatted comparison table
print("╔══════════════╦════════╦════════╦════════╦════════╦═══════════╦═══════════╗")
print("║ Model        ║ MAPE % ║ RMSE   ║ MAE    ║ R²     ║ Train (s) ║ Infer(ms) ║")
print("╠══════════════╬════════╬════════╬════════╬════════╬═══════════╬═══════════╣")

for _, row in comparison_df.iterrows():
    print(f"║ {row['Model']:<12} ║ {row['MAPE %']:>6.2f} ║ {row['RMSE']:>6.2f} ║ "
          f"{row['MAE']:>6.2f} ║ {row['R²']:>6.4f} ║ {row['Train (s)']:>9.2f} ║ "
          f"{row['Infer (ms)']:>9.2f} ║")

print("╚══════════════╩════════╩════════╩════════╩════════╩═══════════╩═══════════╝")

# Also display as DataFrame
print("\n")
display(comparison_df.style.highlight_min(subset=['MAPE %', 'RMSE', 'MAE'], color='#d4edda')
        .highlight_max(subset=['R²'], color='#d4edda')
        .set_caption('Model Comparison — Coal Demand Forecasting'))

╔══════════════╦════════╦════════╦════════╦════════╦═══════════╦═══════════╗
║ Model        ║ MAPE % ║ RMSE   ║ MAE    ║ R²     ║ Train (s) ║ Infer(ms) ║
╠══════════════╬════════╬════════╬════════╬════════╬═══════════╬═══════════╣
║ ARIMA        ║  13.04 ║  34.95 ║  27.70 ║ -0.0988 ║      9.89 ║      5.84 ║
║ Prophet      ║   8.14 ║  26.37 ║  18.26 ║ 0.3744 ║      0.20 ║     27.09 ║
║ LSTM         ║  12.82 ║  34.55 ║  27.36 ║ -0.0736 ║     11.54 ║     14.20 ║
║ XGBoost      ║   6.79 ║  22.09 ║  15.56 ║ 0.5613 ║      4.23 ║      0.24 ║
╚══════════════╩════════╩════════╩════════╩════════╩═══════════╩═══════════╝




,Model,MAPE %,RMSE,MAE,R²,Train (s),Infer (ms)
0,ARIMA,13.040000,34.950000,27.700000,-0.098800,9.890000,5.840000
1,Prophet,8.140000,26.370000,18.260000,0.374400,0.200000,27.090000
2,LSTM,12.820000,34.550000,27.360000,-0.073600,11.540000,14.200000
3,XGBoost,6.790000,22.090000,15.560000,0.561300,4.230000,0.240000


## 5.4 Target Achievement Check

Checking each model against the project targets:
- **MAPE < 8%** — Acceptable forecast accuracy
- **RMSE < 50 tonnes** — Maximum allowable error magnitude

In [5]:
print(f"\nTargets: MAPE < {TARGET_MAPE}%, RMSE < {TARGET_RMSE} tonnes")
print(f"{'='*50}")

for _, row in comparison_df.iterrows():
    mape_ok = '✅' if row['MAPE %'] < TARGET_MAPE else '❌'
    rmse_ok = '✅' if row['RMSE'] < TARGET_RMSE else '❌'
    both_ok = '⭐ MEETS BOTH TARGETS' if (row['MAPE %'] < TARGET_MAPE and row['RMSE'] < TARGET_RMSE) else ''
    print(f"  {row['Model']:<12}  MAPE {mape_ok} ({row['MAPE %']:>6.2f}%)   RMSE {rmse_ok} ({row['RMSE']:>6.2f})  {both_ok}")

# Best model
best_idx = comparison_df['MAPE %'].idxmin()
best_model_name = comparison_df.loc[best_idx, 'Model']
best_mape = comparison_df.loc[best_idx, 'MAPE %']
best_rmse = comparison_df.loc[best_idx, 'RMSE']
best_result = next(r for r in all_results if r['model_name'] == best_model_name)

print(f"\n🏆 Best Model: {best_model_name} (MAPE={best_mape}%, RMSE={best_rmse})")


Targets: MAPE < 8.0%, RMSE < 50.0 tonnes
  ARIMA         MAPE ❌ ( 13.04%)   RMSE ✅ ( 34.95)  
  Prophet       MAPE ❌ (  8.14%)   RMSE ✅ ( 26.37)  
  LSTM          MAPE ❌ ( 12.82%)   RMSE ✅ ( 34.55)  
  XGBoost       MAPE ✅ (  6.79%)   RMSE ✅ ( 22.09)  ⭐ MEETS BOTH TARGETS

🏆 Best Model: XGBoost (MAPE=6.79%, RMSE=22.09)


## 5.5 Visual Comparison — All Models

Plotting all 4 models' predictions against actual values for the test period.

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
colors = ['#E74C3C', '#3498DB', '#9B59B6', '#27AE60']

for idx, (result, color) in enumerate(zip(all_results, colors)):
    ax = axes[idx // 2, idx % 2]
    actuals = result['actuals']
    preds = result['predictions']
    min_len = min(len(actuals), len(preds))
    
    ax.plot(range(min_len), actuals[:min_len], label='Actual', color='#2C3E50', linewidth=1.5)
    ax.plot(range(min_len), preds[:min_len], label=f'{result["model_name"]}', 
            color=color, linewidth=1.5, linestyle='--')
    
    metrics = compute_metrics(actuals[:min_len], preds[:min_len])
    ax.set_title(f'{result["model_name"]} — MAPE: {metrics["MAPE"]:.2f}% | RMSE: {metrics["RMSE"]:.2f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Test Sample Index')
    ax.set_ylabel('Coal (tonnes)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('All Models — Actual vs Predicted Comparison', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5.6 Best Model Forecast — Detailed View

In [7]:
actuals = best_result['actuals']
preds = best_result['predictions']
min_len = min(len(actuals), len(preds))
actuals = actuals[:min_len]
preds = preds[:min_len]

test_dates = pd.to_datetime(test_df['date'].values[:min_len])

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_dates, actuals, label='Actual', color='#2C3E50', linewidth=2)
ax.plot(test_dates, preds, label=f'{best_model_name} Predicted', 
        color='#E74C3C', linewidth=2, linestyle='--')
ax.set_title(f'Best Model ({best_model_name}) — Actual vs Predicted', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Coal Consumption (tonnes)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.7 Prediction Intervals (95% Confidence)

In [8]:
# Compute prediction intervals
if 'lower_bound' in best_result and 'upper_bound' in best_result:
    lower = best_result['lower_bound'][:min_len]
    upper = best_result['upper_bound'][:min_len]
else:
    residuals = actuals - preds
    std_resid = np.std(residuals)
    lower = preds - 1.96 * std_resid
    upper = preds + 1.96 * std_resid

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_dates, actuals, label='Actual', color='#2C3E50', linewidth=2)
ax.plot(test_dates, preds, label='Predicted', color='#E74C3C', linewidth=2)
ax.fill_between(test_dates, lower, upper, alpha=0.2, color='#E74C3C', label='95% Prediction Interval')
ax.set_title('Prediction Intervals (95% Confidence)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Coal Consumption (tonnes)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Coverage
within_bounds = np.sum((actuals >= lower) & (actuals <= upper))
coverage = within_bounds / len(actuals) * 100
print(f"Prediction interval coverage: {coverage:.1f}% ({within_bounds}/{len(actuals)} samples)")

Prediction interval coverage: 94.4% (151/160 samples)


## 5.8 Economic Impact Analysis (in INR)

Quantifying the **business value** of improved forecasting — addressing professor feedback on economic impact.

### Cost Assumptions:
| Parameter | Value | Source |
|-----------|-------|--------|
| Holding cost | ₹500/tonne/day | Industry estimate |
| Shortage cost | ₹8,000/MWh lost | Grid penalty rate |
| Avg MWh lost per shortage | 200 MWh | Historical average |
| Expected inventory reduction | 15% | From improved forecasting |

In [9]:
# Economic impact computation
excess = preds - actuals  # positive = overestimate (excess stock)
excess_positives = excess[excess > 0]
avg_excess_tonnes = float(np.mean(excess_positives)) if len(excess_positives) > 0 else 0
excess_inventory_days = len(excess_positives)

reduction_factor = INVENTORY_REDUCTION_PERCENT / 100.0
holding_cost_saved = avg_excess_tonnes * HOLDING_COST_PER_TONNE_PER_DAY * excess_inventory_days * reduction_factor
test_days = min_len
annual_holding_saved = holding_cost_saved * (DAYS_PER_YEAR / test_days)

shortages = excess[excess < 0]
shortage_events = len(shortages)
events_prevented = int(shortage_events * reduction_factor)
shortage_cost_avoided = events_prevented * AVG_MWH_LOST_PER_SHORTAGE * SHORTAGE_COST_PER_MWH
annual_shortage_saved = shortage_cost_avoided * (DAYS_PER_YEAR / test_days)

total_annual_saving = annual_holding_saved + annual_shortage_saved

# Display
print("╔══════════════════════════════════════════════════════════════════╗")
print("║                    ECONOMIC IMPACT ANALYSIS                      ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║                                                                  ║")
print(f"║  Excess Inventory Days (test period):  {excess_inventory_days:>8}                   ║")
print(f"║  Avg Excess Inventory (tonnes):        {avg_excess_tonnes:>10.2f}                 ║")
print("║                                                                  ║")
print("║  HOLDING COST SAVED:                                             ║")
print(f"║    Excess × Rs {HOLDING_COST_PER_TONNE_PER_DAY:.0f}/tonne/day × {INVENTORY_REDUCTION_PERCENT:.0f}% reduction            ║")
print(f"║    Annual Savings:               Rs {annual_holding_saved:>14,.2f}            ║")
print("║                                                                  ║")
print("║  SHORTAGE EVENTS:                                                ║")
print(f"║    Total shortage risk days:       {shortage_events:>8}                       ║")
print(f"║    Events prevented ({INVENTORY_REDUCTION_PERCENT:.0f}%):          {events_prevented:>8}                       ║")
print(f"║    Cost per event: {AVG_MWH_LOST_PER_SHORTAGE:.0f} MWh × Rs {SHORTAGE_COST_PER_MWH:,.0f}/MWh                        ║")
print(f"║    Annual Savings:               Rs {annual_shortage_saved:>14,.2f}            ║")
print("║                                                                  ║")
print("║  ════════════════════════════════════════════════════════════     ║")
print(f"║  TOTAL ANNUAL SAVINGS:           Rs {total_annual_saving:>14,.2f}            ║")
print("║  ════════════════════════════════════════════════════════════     ║")
print("║                                                                  ║")
print("╚══════════════════════════════════════════════════════════════════╝")

print(f"\n💰 Estimated Annual Savings: Rs {total_annual_saving:,.0f}")
print(f"   (~Rs {total_annual_saving/10000000:.2f} Crore per year)")

╔══════════════════════════════════════════════════════════════════╗
║                    ECONOMIC IMPACT ANALYSIS                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  Excess Inventory Days (test period):       101                   ║
║  Avg Excess Inventory (tonnes):             13.55                 ║
║                                                                  ║
║  HOLDING COST SAVED:                                             ║
║    Excess × Rs 500/tonne/day × 15% reduction            ║
║    Annual Savings:               Rs     234,232.32            ║
║                                                                  ║
║  SHORTAGE EVENTS:                                                ║
║    Total shortage risk days:             59                       ║
║    Events prevented (15%):                 8                       ║
║    Cost per event: 200 MWh × Rs 8,000/M

## 5.9 Residual Analysis

In [10]:
residuals = actuals - preds

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residual distribution
axes[0].hist(residuals, bins=30, color='#3498DB', edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].set_title(f'{best_model_name} — Residual Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

# Residuals over time
axes[1].plot(test_dates, residuals, color='#E74C3C', alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Residuals Over Time', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Residual')
axes[1].grid(True, alpha=0.3)

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot (Normality Check)', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Residual Analysis — {best_model_name}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Residual Statistics:")
print(f"  Mean: {np.mean(residuals):.4f} (should be ≈ 0)")
print(f"  Std:  {np.std(residuals):.4f}")
print(f"  Skew: {pd.Series(residuals).skew():.4f}")

Residual Statistics:
  Mean: -1.5568 (should be ≈ 0)
  Std:  22.0314
  Skew: 1.7282


## 5.10 Final Summary

In [11]:
print("╔══════════════════════════════════════════════════╗")
print("║          EVALUATION SUMMARY                      ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Models Evaluated:        4                      ║")
print(f"║  Best Model:         {best_model_name:<20}     ║")
print(f"║  Best MAPE:          {best_mape:>8.2f}%                  ║")
print(f"║  Best RMSE:          {best_rmse:>8.2f} tonnes            ║")
print(f"║  MAPE Target (<8%):  {'✅ MET' if best_mape < 8 else '❌ NOT MET':<20}     ║")
print(f"║  RMSE Target (<50):  {'✅ MET' if best_rmse < 50 else '❌ NOT MET':<20}     ║")
print(f"║  Annual Savings:     Rs {total_annual_saving:>12,.0f}         ║")
print("╚══════════════════════════════════════════════════╝")

print(f"\n✅ Evaluation complete. Proceed to Notebook 06 for model monitoring.")

╔══════════════════════════════════════════════════╗
║          EVALUATION SUMMARY                      ║
╠══════════════════════════════════════════════════╣
║  Models Evaluated:        4                      ║
║  Best Model:         XGBoost                  ║
║  Best MAPE:              6.79%                  ║
║  Best RMSE:             22.09 tonnes            ║
║  MAPE Target (<8%):  ✅ MET                    ║
║  RMSE Target (<50):  ✅ MET                    ║
║  Annual Savings:     Rs   29,434,232         ║
╚══════════════════════════════════════════════════╝

✅ Evaluation complete. Proceed to Notebook 06 for model monitoring.
